In [ ]:
# ============================================================
# v6.2_hypervgg16_vggface — Colab Script (VGGFace backbone; BNNeck + ArcFace; no SE block)
# Goal: ≥97% Val Acc via (BNNeck + MixUp + Wider TTA + Margin Schedule + Deeper FT + kNN)
# - Unzip dataset from Google Drive → /content/dataset
# - Face crop (MediaPipe) fallback resize
# - Backbone: VGG16 (VGGFace, face-pretrained); head: GAP -> FC -> BNNeck -> ArcFace (no SE block)
# - Head: GAP → FC(256, ReLU) → BNNeck(scale=False) → ArcFace(CosineDenseFixed)
# - Training: MixUp(α=0.2) via tf.data + Warmup+Cosine LR (serializable) + staged unfreezing
#              Phase4 opens block3 + higher ArcFace margin (m=0.35), small LR
# - Eval: Dense head (softmax) + kNN (cosine, k=5, T=0.7) + Wider TTA
# - Saves ALL results: /content/drive/MyDrive/Models/v5_1_hypervgg16_resnet50/runs/<RUN_ID>/
# ============================================================

import os, sys, json, math, time, hashlib, platform, subprocess, zipfile
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, asdict
import numpy as np
import cv2
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import Model, layers
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,
                                     Add, GlobalAveragePooling2D, Dense, Dropout,
                                     Reshape, Multiply)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

# ---- VGGFace (face-pretrained) backbone weights ----
VGGFACE_MEAN = np.array([93.5940, 104.7624, 129.1863], dtype=np.float32)  # BGR mean
VGGFACE_WEIGHTS = tf.keras.utils.get_file(
    "rcmalli_vggface_tf_notop_vgg16.h5",
    "https://github.com/rcmalli/keras-vggface/releases/download/v2.0/rcmalli_vggface_tf_notop_vgg16.h5")
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_recall_fscore_support)

# ------------------ Google Drive ------------------
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("[INFO] Drive already mounted or not in Colab:", e)

# ------------------ TensorFlow Addons (EMA optional) ------------------
USE_EMA = False
tfa = None
try:
    import tensorflow_addons as tfa
    USE_EMA = True
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-addons"])
        import tensorflow_addons as tfa
        USE_EMA = True
    except Exception:
        print("[INFO] tensorflow-addons unavailable. Proceeding WITHOUT EMA.")

# ------------------ Config ------------------
ALGO_NAME      = "v6_2_hypervgg16_vggface"
OBJECTIVE      = "≥ 97% Val Accuracy"

DRIVE_BASE_DIR = "/content/drive/MyDrive/Models"
ZIP_PATH       = '/content/drive/MyDrive/Papers/CNN VGG16 with Resnet 50/DS-SID-1to150-20251004T165019Z-1-002.zip'
EXTRACT_ROOT   = Path('/content/dataset')
DATASET_ROOT   = EXTRACT_ROOT / 'DS-SID-1to150'

# ↑ لدفعة دقة إضافية: 256x256
IMG_H, IMG_W   = 256, 256
BATCH_SIZE     = 32
SEED           = 42

# ArcFace schedule
ARC_S          = 32.0
ARC_M_PHASES   = (0.20, 0.25, 0.25, 0.35)  # آخر مرحلة بهامش أعلى

# Training phases (Phase4 = fine-tune block3)
EPOCHS_PHASES  = (6, 6, 8, 8)   # freeze → unfreeze 20% → unfreeze 40% → open block3 (small LR)
WARMUP_EPOCHS  = 1
EMA_DECAY      = 0.999

# TTA
TTA_ROUNDS     = 10
TTA_USE_FLIP   = True

# Latency profiling
LAT_WARMUP = 10
LAT_ITERS  = 50

# ============================================================
# (0) Unzip dataset
# ============================================================
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
if not DATASET_ROOT.exists():
    if not Path(ZIP_PATH).exists():
        raise FileNotFoundError(f"ZIP not found at: {ZIP_PATH}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_ROOT)
    print(f"[OK] Unzipped → {EXTRACT_ROOT}")
else:
    print(f"[Skip] Already extracted: {DATASET_ROOT}")
print("[DIR] extract_root contents:", os.listdir(EXTRACT_ROOT)[:10])
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Expected dataset folder not found: {DATASET_ROOT}")

# ============================================================
# Utils
# ============================================================
def fix_seeds(seed=42):
    import random
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def env_info():
    return {
        "python": sys.version,
        "platform": platform.platform(),
        "tensorflow": tf.__version__,
        "numpy": np.__version__,
        "sklearn": __import__("sklearn").__version__,
        "cv2": cv2.__version__,
        "gpu": [str(d) for d in tf.config.list_physical_devices('GPU')],
        "use_ema": USE_EMA,
    }

def short_hash(d: dict, n=8):
    return hashlib.md5(json.dumps(d, sort_keys=True).encode("utf-8")).hexdigest()[:n]

def make_run_dirs(base_drive_dir: str, algo_name: str, cfg: dict):
    base = Path(base_drive_dir) / algo_name
    base.mkdir(parents=True, exist_ok=True)
    run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + "_" + short_hash(cfg)
    run = base / "runs" / run_id
    dirs = {
        "base": base, "run": run,
        "config": run / "config",
        "metrics": run / "metrics",
        "curves": run / "metrics" / "curves",
        "preds": run / "predictions",
        "reports": run / "reports",
        "confusion": run / "reports" / "confusion",
        "performance": run / "performance",
        "checks": run / "checkpoints",
        "latest": base / "latest",
        "proto": run / "prototypes",
    }
    for p in dirs.values(): p.mkdir(parents=True, exist_ok=True)
    with open(dirs["latest"] / "RUN_ID", "w", encoding="utf-8") as f:
        f.write(run_id + "\n")
    return dirs, run_id

# ============================================================
# Optional Face Crop via MediaPipe (fallback to resize)
# ============================================================
def ensure_mediapipe():
    try:
        import mediapipe as mp  # noqa
        return True
    except Exception:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mediapipe"])
            import mediapipe as mp  # noqa
            return True
        except Exception as e:
            print("[WARN] MediaPipe not available, using simple resize. Err:", e)
            return False

MP_READY = ensure_mediapipe()

def face_crop(img, target_size=(256,256)):
    if not MP_READY:
        return cv2.resize(img, target_size)
    try:
        import mediapipe as mp
        mp_fd = mp.solutions.face_detection
        with mp_fd.FaceDetection(model_selection=1, min_detection_confidence=0.4) as fd:
            rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            res = fd.process(rgb)
            if not res.detections:
                return cv2.resize(img, target_size)
            det = max(res.detections, key=lambda d: d.score[0])
            bbox = det.location_data.relative_bounding_box
            h, w = img.shape[:2]
            x1 = max(int(bbox.xmin * w) - 10, 0)
            y1 = max(int(bbox.ymin * h) - 10, 0)
            x2 = min(int((bbox.xmin + bbox.width) * w) + 10, w)
            y2 = min(int((bbox.ymin + bbox.height) * h) + 10, h)
            face = img[y1:y2, x1:x2]
            if face.size == 0: return cv2.resize(img, target_size)
            return cv2.resize(face, target_size)
    except Exception:
        return cv2.resize(img, target_size)

# ============================================================
# Data Loading + preprocess_input
# ============================================================
def load_dataset(root: Path, size=(256,256)):
    classes = sorted([d.name for d in root.iterdir() if d.is_dir() and d.name.isdigit()], key=lambda x:int(x))
    if not classes:
        raise RuntimeError(f"No numeric class folders under {root}")
    f2l = {name:i for i,name in enumerate(classes)}
    X, y = [], []
    for cls in classes:
        lab = f2l[cls]
        for fname in os.listdir(root/cls):
            path = str(root/cls/fname)
            img = cv2.imread(path)
            if img is None: continue
            img = face_crop(img, size)
            X.append(img); y.append(lab)
    X = np.asarray(X, dtype=np.float32)
    X = X - VGGFACE_MEAN  # VGGFace (BGR) mean subtraction (replaces ImageNet preprocess_input)
    labels = np.asarray(y, dtype=np.int32)
    y = tf.keras.utils.to_categorical(labels, num_classes=len(classes)).astype('float32')
    return X, y, f2l

# ============================================================
# MixUp + tf.data + RandAug-light
# ============================================================
def make_augmenter():
    return tf.keras.Sequential([
        tf.keras.layers.RandomBrightness(0.08),
        tf.keras.layers.RandomContrast(0.08),
        tf.keras.layers.RandomRotation(0.05),
        tf.keras.layers.RandomTranslation(0.03, 0.03),
        tf.keras.layers.RandomZoom(0.05),
    ], name="randaug_light")

def mixup_batch(x, y, alpha=0.2):
    x = tf.cast(x, tf.float32)
    y = tf.cast(y, tf.float32)
    g1 = tf.random.gamma(shape=[tf.shape(x)[0],1,1,1], alpha=alpha, dtype=tf.float32)
    g2 = tf.random.gamma(shape=[tf.shape(x)[0],1,1,1], alpha=alpha, dtype=tf.float32)
    lam = g1 / (g1 + g2)
    lam_y = tf.reshape(lam, [-1,1])
    idx = tf.random.shuffle(tf.range(tf.shape(x)[0]))
    x2, y2 = tf.gather(x, idx), tf.gather(y, idx)
    x_mix = lam * x + (1.0 - lam) * x2
    y_mix = lam_y * y + (1.0 - lam_y) * y2
    return x_mix, y_mix

def build_tfdata(X, Y, batch_size, training=True, shuffle=True, mixup_alpha=0.2):
    ds = tf.data.Dataset.from_tensor_slices((X, Y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(lambda x,y: (tf.cast(x, tf.float32), tf.cast(y, tf.float32)),
                num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=training)
    aug = make_augmenter()
    if training:
        ds = ds.map(lambda x,y: (aug(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
        if mixup_alpha and mixup_alpha > 0:
            ds = ds.map(lambda x,y: mixup_batch(x,y,alpha=mixup_alpha), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

# ============================================================
# SE + Residual (zero-gamma via built-in 'zeros')
# ============================================================
def se_block(x, r=16, name="se"):
    c = x.shape[-1]
    s = GlobalAveragePooling2D(name=f"{name}_gap")(x)
    s = Dense(max(c//r, 8), activation="relu", name=f"{name}_fc1")(s)
    s = Dense(c, activation="sigmoid", name=f"{name}_fc2")(s)
    s = Reshape((1,1,c))(s)
    return Multiply(name=f"{name}_scale")([x, s])

def bottleneck_res_se(x, filters, name="res1"):
    f1,f2,f3 = filters
    sc = x
    y = Conv2D(f1,1,padding='same',use_bias=False,name=f"{name}_conv1")(x)
    y = BatchNormalization(name=f"{name}_bn1")(y); y = Activation('relu',name=f"{name}_relu1")(y)
    y = Conv2D(f2,3,padding='same',use_bias=False,name=f"{name}_conv2")(y)
    y = BatchNormalization(name=f"{name}_bn2")(y); y = Activation('relu',name=f"{name}_relu2")(y)
    y = Conv2D(f3,1,padding='same',use_bias=False,name=f"{name}_conv3")(y)
    y = BatchNormalization(gamma_initializer='zeros', name=f"{name}_bn3")(y)
    if sc.shape[-1] != f3:
        sc = Conv2D(f3,1,padding='same',use_bias=False,name=f"{name}_proj")(sc)
        sc = BatchNormalization(name=f"{name}_proj_bn")(sc)
    y = Add(name=f"{name}_add")([y, sc])
    y = Activation('relu', name=f"{name}_out_relu")(y)
    y = se_block(y, r=16, name=f"{name}_se")
    return y

# ============================================================
# ArcFace Head (CosineDenseFixed) + Serializable Loss
# ============================================================
for _name in ["CosineDense", "CosineDenseFixed"]:
    try: del globals()[_name]
    except KeyError: pass

class CosineDenseFixed(layers.Layer):
    def __init__(self, units, s=32.0, name="cosine_dense", **kwargs):
        super().__init__(name=name, **kwargs)
        self.units = int(units); self.s = float(s)
    def build(self, input_shape):
        in_dim = int(input_shape[-1])
        self.W = self.add_weight(name="W", shape=(in_dim, self.units),
                                 initializer="glorot_uniform", trainable=True)
    def call(self, x):
        x = tf.nn.l2_normalize(x, axis=-1)
        W = tf.nn.l2_normalize(self.W, axis=0)
        return self.s * tf.matmul(x, W)

def arcface_logits(logits, y_true, s=32.0, m=0.2):
    cos = logits / s
    sin = tf.sqrt(tf.maximum(1.0 - tf.square(cos), 1e-7))
    cos_m, sin_m = tf.cos(m), tf.sin(m)
    cos_theta_m = cos * cos_m - sin * sin_m
    y_true = tf.cast(y_true, cos.dtype)
    logits_m = s * (cos_theta_m * y_true + cos * (1.0 - y_true))
    return logits_m

from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable(package="Custom")
class ArcFaceLoss(tf.keras.losses.Loss):
    def __init__(self, s=32.0, m=0.2, name="arcface_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.s = float(s); self.m = float(m)
    def call(self, y_true, y_pred):
        logits_m = arcface_logits(y_pred, y_true, s=self.s, m=self.m)
        return tf.keras.losses.categorical_crossentropy(y_true, logits_m, from_logits=True)
    def get_config(self):
        return {"s": self.s, "m": self.m, "name": self.name}

# ============================================================
# Warmup+Cosine (serializable)
# ============================================================
class WarmupCosine(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base_lr, steps_per_epoch, total_epochs, warmup_epochs=1, min_lr=1e-6):
        super().__init__()
        self.base_lr_f        = float(base_lr)
        self.steps_per_epoch_f= int(steps_per_epoch)
        self.total_epochs_f   = int(total_epochs)
        self.warmup_epochs_f  = int(warmup_epochs)
        self.min_lr_f         = float(min_lr)
        self.base_lr  = tf.convert_to_tensor(self.base_lr_f, dtype=tf.float32)
        self.spe      = tf.cast(self.steps_per_epoch_f, tf.float32)
        self.T        = tf.cast(self.total_epochs_f, tf.float32) * self.spe
        self.wu       = tf.cast(self.warmup_epochs_f, tf.float32) * self.spe
        self.min_lr   = tf.convert_to_tensor(self.min_lr_f, dtype=tf.float32)
        self.pi       = tf.constant(np.pi, dtype=tf.float32)
    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        def warm():
            return self.base_lr * (step / tf.maximum(self.wu, 1.0))
        def cosine():
            t = tf.maximum(step - self.wu, 0.0)
            T = tf.maximum(self.T - self.wu, 1.0)
            return self.min_lr + 0.5*(self.base_lr - self.min_lr)*(1.0 + tf.cos(self.pi*t/T))
        return tf.cond(self.wu > 0.0,
                       lambda: tf.cond(step < self.wu, warm, cosine),
                       cosine)
    def get_config(self):
        return {"base_lr": self.base_lr_f, "steps_per_epoch": self.steps_per_epoch_f,
                "total_epochs": self.total_epochs_f, "warmup_epochs": self.warmup_epochs_f,
                "min_lr": self.min_lr_f}
    @classmethod
    def from_config(cls, config): return cls(**config)

# ============================================================
# Optimizer
# ============================================================
def make_optimizer(base_lr, steps_per_epoch, total_epochs, wd=1e-4):
    lr_sched = WarmupCosine(base_lr, steps_per_epoch, total_epochs, warmup_epochs=WARMUP_EPOCHS, min_lr=1e-6)
    base_opt = tf.keras.optimizers.AdamW(learning_rate=lr_sched, weight_decay=wd)
    if USE_EMA and tfa is not None:
        return tfa.optimizers.MovingAverage(base_opt, average_decay=EMA_DECAY, ema_overwrite_frequency=None)
    return base_opt

# ============================================================
# Build v5.1 model  (BNNeck + ArcFace)
# ============================================================
def build_model(input_shape, num_classes, freeze_base=True):
    aug = tf.keras.Sequential([
        tf.keras.layers.RandomBrightness(0.05),
        tf.keras.layers.RandomContrast(0.05),
        tf.keras.layers.RandomRotation(0.03),
        tf.keras.layers.RandomTranslation(0.02, 0.02),
    ], name="aug")

    inp = Input(shape=input_shape)
    x = aug(inp)
    base = VGG16(include_top=False, weights=None, input_tensor=x)
    base.load_weights(VGGFACE_WEIGHTS)  # face-pretrained (VGGFace) backbone
    if freeze_base:
        for l in base.layers: l.trainable = False

    x = base.output
    x = GlobalAveragePooling2D(name="gap")(x)
    feat = Dense(256, activation='relu', name='fc1')(x)
    bnneck = BatchNormalization(scale=False, name="bnneck")(feat)
    logits = CosineDenseFixed(num_classes, s=ARC_S, name="cosine_dense")(bnneck)
    model = Model(inp, logits, name="v6_2_hypervgg16_vggface")
    return model

# ============================================================
# Compile + Train phases (with margin schedule)
# ============================================================
def compile_with_schedule(model, base_lr, steps_per_epoch, total_epochs, margin, weight_decay=1e-4):
    opt = make_optimizer(base_lr, steps_per_epoch, total_epochs, wd=weight_decay)
    loss = ArcFaceLoss(s=ARC_S, m=margin)
    model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

def set_trainable_layers_fraction(model, fraction=0.2):
    trainables = int(len(model.layers) * fraction)
    for l in model.layers[-trainables:]:
        if not isinstance(l, BatchNormalization):
            l.trainable = True

def unfreeze_block(model, block_name):
    for l in model.layers:
        if block_name in l.name and not isinstance(l, BatchNormalization):
            l.trainable = True

def train_phases(model, ds_train, ds_val, steps_per_epoch, dirs):
    ckpt_path = str(dirs["checks"] / "best.keras")
    ckpt = ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True, verbose=1)
    early = EarlyStopping(monitor='val_accuracy', patience=3, min_delta=1e-3, restore_best_weights=True, verbose=1)

    # Phase 1: freeze backbone, m=ARC_M_PHASES[0], lr=1e-3
    compile_with_schedule(model, base_lr=1e-3, steps_per_epoch=steps_per_epoch,
                          total_epochs=EPOCHS_PHASES[0], margin=ARC_M_PHASES[0], weight_decay=1e-4)
    h1 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_PHASES[0], callbacks=[ckpt, early], verbose=1)

    # Phase 2: unfreeze last 20%, m=ARC_M_PHASES[1], lr=3e-4
    set_trainable_layers_fraction(model, fraction=0.2)
    compile_with_schedule(model, base_lr=3e-4, steps_per_epoch=steps_per_epoch,
                          total_epochs=EPOCHS_PHASES[1], margin=ARC_M_PHASES[1], weight_decay=2e-4)
    h2 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_PHASES[1], callbacks=[ckpt, early], verbose=1)

    # Phase 3: unfreeze last 40%, m=ARC_M_PHASES[2], lr=1e-4
    set_trainable_layers_fraction(model, fraction=0.4)
    compile_with_schedule(model, base_lr=1e-4, steps_per_epoch=steps_per_epoch,
                          total_epochs=EPOCHS_PHASES[2], margin=ARC_M_PHASES[2], weight_decay=3e-4)
    h3 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_PHASES[2], callbacks=[ckpt, early], verbose=1)

    # Phase 4: open block3, m=0.35, lr=5e-5 (fine-tune deeper)
    unfreeze_block(model, 'block3')
    compile_with_schedule(model, base_lr=5e-5, steps_per_epoch=steps_per_epoch,
                          total_epochs=EPOCHS_PHASES[3], margin=ARC_M_PHASES[3], weight_decay=3e-4)
    h4 = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_PHASES[3], callbacks=[ckpt, early], verbose=1)

    # Save curves
    def save_hist(hist, prefix):
        for k,v in hist.history.items():
            with open(dirs["curves"]/f"{prefix}_{k}.csv","w",encoding="utf-8") as f:
                f.write("epoch,value\n")
                for i, val in enumerate(v,1): f.write(f"{i},{val}\n")
    save_hist(h1,"phase1"); save_hist(h2,"phase2"); save_hist(h3,"phase3"); save_hist(h4,"phase4")

    # Load best
    if os.path.exists(ckpt_path):
        model = tf.keras.models.load_model(
            ckpt_path,
            custom_objects={"CosineDenseFixed": CosineDenseFixed,
                            "WarmupCosine": WarmupCosine,
                            "ArcFaceLoss": ArcFaceLoss}
        )
    return model

# ============================================================
# Embedding extractor + kNN (cosine, k=5, T=0.7)
# ============================================================
def build_embedder_from_model(model):
    inp = model.input
    bnneck_layer = model.get_layer("bnneck").output
    return Model(inp, bnneck_layer, name="embedder_v5_1")

def l2_normalize(a, axis=-1, eps=1e-9):
    return a / (np.linalg.norm(a, axis=axis, keepdims=True) + eps)

def cosine_sim(a, b):
    a = l2_normalize(a); b = l2_normalize(b)
    return np.dot(a, b.T)

def knn_predict(train_emb, train_labels, val_emb, k=5, temp=0.7):
    # cosine sims → choose top-k neighbors with soft weights
    sims = cosine_sim(val_emb, train_emb)  # [N_val, N_train]
    N_val, N_train = sims.shape
    preds = np.zeros((N_val,), dtype=np.int32)
    for i in range(N_val):
        idx = np.argpartition(-sims[i], k)[:k]
        s = sims[i, idx] / max(temp, 1e-6)      # temperature scaling
        w = np.exp(s - s.max())                 # softmax stable
        w = w / (w.sum() + 1e-9)
        labs = train_labels[idx]
        # vote by weighted sum per class
        unique, inv = np.unique(labs, return_inverse=True)
        scores = np.zeros((unique.shape[0],), dtype=np.float32)
        np.add.at(scores, inv, w)
        preds[i] = unique[np.argmax(scores)]
    return preds

# ============================================================
# Wider TTA for arrays
# ============================================================
def make_tta_aug():
    return tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.07),
        tf.keras.layers.RandomTranslation(0.02, 0.02),
        tf.keras.layers.RandomZoom(0.04),
        tf.keras.layers.RandomBrightness(0.05),
        tf.keras.layers.RandomContrast(0.05),
    ], name="tta_aug")

def tta_predict_logits(model, X, rounds=10, use_flip=True):
    logits_list = []
    tta = make_tta_aug()
    logits_base = model.predict(X, batch_size=BATCH_SIZE, verbose=0)
    logits_list.append(logits_base)
    if use_flip:
        Xf = X[:, :, ::-1, :]
        logits_list.append(model.predict(Xf, batch_size=BATCH_SIZE, verbose=0))
    for _ in range(rounds):
        Xa = tta(X, training=True).numpy()
        logits_list.append(model.predict(Xa, batch_size=BATCH_SIZE, verbose=0))
    logits = np.mean(np.stack(logits_list, axis=0), axis=0)
    return logits

# ============================================================
# Evaluate + Log (Dense & kNN)
# ============================================================
def evaluate_and_log(model, X_train, y_train, X_val, y_val, dirs, f2l):
    # Dense head with wider TTA
    logits = tta_predict_logits(model, X_val, rounds=TTA_ROUNDS, use_flip=TTA_USE_FLIP)
    probs  = tf.nn.softmax(logits).numpy()
    y_true = np.argmax(y_val, axis=1)
    y_pred_dense = np.argmax(probs, axis=1)
    acc_dense = accuracy_score(y_true, y_pred_dense)

    # kNN on BNNeck
    embedder = build_embedder_from_model(model)
    E_tr = embedder.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
    E_tr = l2_normalize(E_tr)
    y_tr_ids = np.argmax(y_train, axis=1)

    E_val = embedder.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    E_val = l2_normalize(E_val)
    y_pred_knn = knn_predict(E_tr, y_tr_ids, E_val, k=5, temp=0.7)
    acc_knn = accuracy_score(y_true, y_pred_knn)

    # Reports
    rep_dense = classification_report(y_true, y_pred_dense, digits=4, zero_division=0)
    with open(dirs["reports"]/ "classification_report_dense.txt","w",encoding="utf-8") as f:
        f.write(rep_dense+"\n")
    rep_knn = classification_report(y_true, y_pred_knn, digits=4, zero_division=0)
    with open(dirs["reports"]/ "classification_report_knn.txt","w",encoding="utf-8") as f:
        f.write(rep_knn+"\n")

    inv = {v:k for k,v in f2l.items()}

    # confusion (dense)
    cm = confusion_matrix(y_true, y_pred_dense, labels=range(y_val.shape[1]))
    plt.figure(figsize=(7,6))
    plt.imshow(cm, interpolation='nearest'); plt.title("Confusion Matrix (VAL) — Dense")
    plt.colorbar(); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(dirs["confusion"]/ "confmat_dense.png", dpi=160); plt.close()

    # confusion (kNN)
    cm2 = confusion_matrix(y_true, y_pred_knn, labels=range(y_val.shape[1]))
    plt.figure(figsize=(7,6))
    plt.imshow(cm2, interpolation='nearest'); plt.title("Confusion Matrix (VAL) — kNN")
    plt.colorbar(); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.savefig(dirs["confusion"]/ "confmat_knn.png", dpi=160); plt.close()

    # save preds
    np.savez_compressed(dirs["preds"]/ "val_preds_dense.npz", logits=logits, probs=probs, y_true=y_true)
    np.savez_compressed(dirs["preds"]/ "val_preds_knn.npz", y_pred=y_pred_knn, y_true=y_true)

    with open(dirs["metrics"]/ "metrics.json","w",encoding="utf-8") as f:
        json.dump({
            "val_acc_dense_tta": float(acc_dense),
            "val_acc_knn": float(acc_knn),
            "num_classes": int(y_val.shape[1]),
            "input_size": [IMG_H, IMG_W, 3],
            "tta_rounds": int(TTA_ROUNDS),
            "tta_flip": bool(TTA_USE_FLIP)
        }, f, ensure_ascii=False, indent=2)

    return acc_dense, acc_knn

def measure_latency(model, dirs, input_shape=(256,256,3), warmup=10, iters=50):
    x = np.random.randn(1,*input_shape).astype(np.float32)
    x = preprocess_input(x.copy())
    for _ in range(warmup): _ = model.predict(x, verbose=0)
    times=[]
    for _ in range(iters):
        t0=time.perf_counter(); _=model.predict(x, verbose=0); t1=time.perf_counter()
        times.append((t1-t0)*1000.0)
    times=np.array(times)
    with open(dirs["performance"]/ "latency.csv","w",encoding="utf-8") as f:
        f.write("iter,ms\n")
        for i,v in enumerate(times,1): f.write(f"{i},{v:.3f}\n")
    return {"mean_ms":float(times.mean()),
            "p50_ms":float(np.percentile(times,50)),
            "p90_ms":float(np.percentile(times,90)),
            "p99_ms":float(np.percentile(times,99))}

# ============================================================
# Main
# ============================================================
@dataclass
class TrainConfig:
    algo_name: str
    objective: str
    drive_base: str
    dataset_root: str
    img_h: int
    img_w: int
    batch_size: int
    seed: int
    epochs: tuple
    arc_s: float
    arc_m_phases: tuple
    ema_decay: float
    warmup_epochs: int
    use_ema: bool
    mixup_alpha: float
    tta_rounds: int
    tta_flip: bool

fix_seeds(SEED)
assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"

# Load arrays
X, y, f2l = load_dataset(DATASET_ROOT, (IMG_W, IMG_H))
y_ids = np.argmax(y, axis=1)

x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y_ids
)

# tf.data with MixUp
ds_train = build_tfdata(x_train, y_train, BATCH_SIZE, training=True,  shuffle=True,  mixup_alpha=0.2)
ds_val   = build_tfdata(x_val,   y_val,   BATCH_SIZE, training=False, shuffle=False, mixup_alpha=0.0)
steps_per_epoch = math.ceil(len(x_train)/BATCH_SIZE)

cfg = TrainConfig(
    algo_name=ALGO_NAME, objective=OBJECTIVE, drive_base=DRIVE_BASE_DIR,
    dataset_root=str(DATASET_ROOT), img_h=IMG_H, img_w=IMG_W,
    batch_size=BATCH_SIZE, seed=SEED, epochs=EPOCHS_PHASES,
    arc_s=ARC_S, arc_m_phases=ARC_M_PHASES, ema_decay=EMA_DECAY, warmup_epochs=WARMUP_EPOCHS,
    use_ema=USE_EMA, mixup_alpha=0.2, tta_rounds=TTA_ROUNDS, tta_flip=TTA_USE_FLIP
)
dirs, run_id = make_run_dirs(DRIVE_BASE_DIR, ALGO_NAME, asdict(cfg))

# Build + Train
model = build_model((IMG_H, IMG_W, 3), num_classes=y.shape[1], freeze_base=True)
model = train_phases(model, ds_train, ds_val, steps_per_epoch, dirs)

# Evaluate (Dense & kNN)
acc_dense, acc_knn = evaluate_and_log(model, x_train, y_train, x_val, y_val, dirs, f2l)
lat = measure_latency(model, dirs, input_shape=(IMG_H, IMG_W, 3), warmup=LAT_WARMUP, iters=LAT_ITERS)

# Save configs & summary
with open(dirs["config"]/ "train_config.json","w",encoding="utf-8") as f:
    json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)
with open(dirs["config"]/ "env_info.json","w",encoding="utf-8") as f:
    json.dump(env_info(), f, ensure_ascii=False, indent=2)
with open(dirs["run"]/ "run_summary.md","w",encoding="utf-8") as f:
    f.write(f"# {ALGO_NAME} — Run Summary\n\n")
    f.write(f"- **Objective:** {OBJECTIVE}\n")
    f.write(f"- **Input:** {IMG_H}×{IMG_W}\n")
    f.write(f"- **Batch:** {BATCH_SIZE}\n")
    f.write(f"- **Epochs (phases):** {EPOCHS_PHASES}\n")
    f.write(f"- **ArcFace schedule (s={ARC_S}):** m phases={ARC_M_PHASES}\n")
    f.write(f"- **Head:** BNNeck + ArcFace\n")
    f.write(f"- **Aug:** MixUp(α=0.2) + RandAug-light\n")
    f.write(f"- **TTA:** rounds={TTA_ROUNDS}, flip={TTA_USE_FLIP}\n")
    f.write(f"- **Val Acc (Dense + TTA):** {acc_dense:.4f}\n")
    f.write(f"- **Val Acc (kNN):** {acc_knn:.4f}\n")
    f.write(f"- **Latency (ms):** mean={lat['mean_ms']:.2f}, p50={lat['p50_ms']:.2f}, "
            f"p90={lat['p90_ms']:.2f}, p99={lat['p99_ms']:.2f}\n")

# Save model
final_model_path = dirs["checks"] / "final_v5_1.keras"
model.save(final_model_path)
print(f"\n[DONE] Saved everything under: {dirs['run']}")
print(f"[MODEL] {final_model_path}")




In [ ]:
# ============================================================
# v6 (VGGFace) — honest validation: 5-fold Stratified CV (mean +/- std)
# Reuses v6 functions from the cell above (load_dataset/build_model now use VGGFace).
# Run AFTER the algorithm cell. WARNING: trains N_FOLDS models -> long.
# ============================================================
import numpy as np, math, gc, tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from dataclasses import asdict
from pathlib import Path

N_FOLDS = 5   # set 3 to shorten
CV_OUT = Path(DRIVE_BASE_DIR) / ALGO_NAME / "cv5"
CV_OUT.mkdir(parents=True, exist_ok=True)

X, y, f2l = load_dataset(DATASET_ROOT, (IMG_W, IMG_H))   # VGGFace preprocessing (v6)
yids = np.argmax(y, axis=1)
print(f"classes={y.shape[1]} | images={len(X)}")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
dense, knn = [], []
for fold, (tr, vl) in enumerate(skf.split(X, yids), 1):
    print("="*70); print(f"[v6 VGGFace] FOLD {fold}/{N_FOLDS}"); print("="*70)
    fix_seeds(42); tf.keras.backend.clear_session()
    x_tr, x_vl = X[tr], X[vl]; y_tr, y_vl = y[tr], y[vl]
    yt = np.argmax(y_vl, axis=1)
    dtr = build_tfdata(x_tr, y_tr, BATCH_SIZE, training=True,  shuffle=True,  mixup_alpha=0.2)
    dvl = build_tfdata(x_vl, y_vl, BATCH_SIZE, training=False, shuffle=False, mixup_alpha=0.0)
    spe = math.ceil(len(x_tr) / BATCH_SIZE)
    cfg = TrainConfig(algo_name=ALGO_NAME, objective=OBJECTIVE, drive_base=DRIVE_BASE_DIR,
        dataset_root=str(DATASET_ROOT), img_h=IMG_H, img_w=IMG_W, batch_size=BATCH_SIZE,
        seed=42, epochs=EPOCHS_PHASES, arc_s=ARC_S, arc_m_phases=ARC_M_PHASES, ema_decay=EMA_DECAY,
        warmup_epochs=WARMUP_EPOCHS, use_ema=USE_EMA, mixup_alpha=0.2,
        tta_rounds=TTA_ROUNDS, tta_flip=TTA_USE_FLIP)
    dirs, _ = make_run_dirs(DRIVE_BASE_DIR, ALGO_NAME, asdict(cfg))
    m = build_model((IMG_H, IMG_W, 3), num_classes=y.shape[1], freeze_base=True)   # VGGFace backbone
    m = train_phases(m, dtr, dvl, spe, dirs)
    logits = tta_predict_logits(m, x_vl, rounds=TTA_ROUNDS, use_flip=TTA_USE_FLIP)
    dense.append(accuracy_score(yt, np.argmax(tf.nn.softmax(logits).numpy(), axis=1)))
    emb = build_embedder_from_model(m)
    E_tr = emb.predict(x_tr, batch_size=BATCH_SIZE, verbose=0)
    E_vl = emb.predict(x_vl, batch_size=BATCH_SIZE, verbose=0)
    knn.append(accuracy_score(yt, knn_predict(E_tr, np.argmax(y_tr, axis=1), E_vl, k=5, temp=0.7)))
    print(f"[v6 VGGFace] FOLD {fold} dense={dense[-1]*100:.2f}%  knn={knn[-1]*100:.2f}%")
    del m, emb; gc.collect(); tf.keras.backend.clear_session()

dense = np.array(dense)*100; knn = np.array(knn)*100
def ms(a): return f"{a.mean():.2f} +/- {a.std(ddof=1):.2f}"
print("\n========= v6 (VGGFace) %d-FOLD CV =========" % N_FOLDS)
print(f"Dense+TTA: {ms(dense)} %   folds={list(np.round(dense,2))}")
print(f"kNN:       {ms(knn)} %   folds={list(np.round(knn,2))}")
print("(baselines: MobileNetV3 93.48, EfficientNetB0 94.15 ; v5/ImageNet 92.92)")
with open(CV_OUT/"v6_cv_summary.txt","w",encoding="utf-8") as f:
    f.write(f"Dense+TTA folds={list(np.round(dense,2))} mean+/-std={ms(dense)}\n")
    f.write(f"kNN folds={list(np.round(knn,2))} mean+/-std={ms(knn)}\n")
print("[SAVED]", CV_OUT)


In [ ]:
# ============================================================
# v6.2 (VGGFace) — Verification metrics: AUC, EER, TAR@FAR
# Run AFTER the algorithm cell (cell 0): uses model, x_val, y_val,
# build_embedder_from_model, l2_normalize, BATCH_SIZE.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from pathlib import Path

VERIF_OUT = Path(DRIVE_BASE_DIR) / ALGO_NAME / "verification"
VERIF_OUT.mkdir(parents=True, exist_ok=True)

embedder = build_embedder_from_model(model)
E = l2_normalize(embedder.predict(x_val, batch_size=BATCH_SIZE, verbose=0))
yv = np.argmax(y_val, axis=1)
N = E.shape[0]
S = E @ E.T
iu = np.triu_indices(N, k=1)
scores = S[iu]
labels = (yv[iu[0]] == yv[iu[1]]).astype(np.int32)
print(f"pairs={len(scores)} | genuine={int(labels.sum())} | impostor={int((labels==0).sum())}")

fpr, tpr, thr = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)
fnr = 1 - tpr
eer_idx = int(np.nanargmin(np.abs(fnr - fpr)))
eer = (fpr[eer_idx] + fnr[eer_idx]) / 2.0

def tar_at_far(ft):
    idx = np.where(fpr <= ft)[0]
    return (None if len(idx) == 0 else tpr[idx[-1]])

print("\n=== v6.2 Verification metrics (validation pairs) ===")
print(f"AUC = {roc_auc:.4f}")
print(f"EER = {eer*100:.2f}%")
for ft in [1e-1, 1e-2, 1e-3]:
    t = tar_at_far(ft)
    print(f"TAR@FAR={ft:g}: " + ("n/a" if t is None else f"{t*100:.2f}%"))

plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"HayderVGR v6.2 (AUC={roc_auc:.3f}, EER={eer*100:.2f}%)")
plt.plot([0,1],[0,1],'--',color='gray')
plt.xlabel("False Acceptance Rate (FAR)"); plt.ylabel("True Acceptance Rate (TAR)")
plt.title("Verification ROC - DS-SID (v6.2)"); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig(VERIF_OUT/"roc_verification_v6_2.png", dpi=160); plt.show()

with open(VERIF_OUT/"verification_v6_2.txt","w",encoding="utf-8") as f:
    f.write(f"AUC={roc_auc:.4f}\nEER={eer*100:.2f}%\n")
    for ft in [1e-1,1e-2,1e-3]:
        t=tar_at_far(ft); f.write(f"TAR@FAR={ft:g}={'n/a' if t is None else f'{t*100:.2f}%'}\n")
print("[SAVED]", VERIF_OUT)


In [ ]:
# ============================================================
# [Comparison 1] Dedicated face model on DS-SID: InsightFace ArcFace (buffalo_l)
# Frozen embedder + cosine k-NN under the SAME 5-fold CV (seed=42) as HayderVGR.
# onnxruntime-based (no torch/torchvision conflict); ArcFace is the same loss
# family as ours -> an apt, fair "off-the-shelf face model vs ours" comparison.
# Run AFTER the algorithm cell (needs DATASET_ROOT).
# ============================================================
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","-q","install","insightface","onnxruntime-gpu"], check=False)

import os, numpy as np, cv2
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from insightface.app import FaceAnalysis

def make_app():
    for prov in (['CUDAExecutionProvider','CPUExecutionProvider'], ['CPUExecutionProvider']):
        try:
            app = FaceAnalysis(name='buffalo_l', providers=prov)
            app.prepare(ctx_id=(0 if 'CUDA' in prov[0] else -1), det_size=(640,640))
            print("InsightFace ready ->", prov[0]); return app
        except Exception as e:
            print("  provider", prov[0], "failed:", type(e).__name__, e)
    raise RuntimeError("InsightFace could not initialize")

app = make_app()
rec = app.models['recognition']   # ArcFace recognition sub-model

def load_raw_paths(root):
    classes = sorted([d.name for d in Path(root).iterdir() if d.is_dir() and d.name.isdigit()],
                     key=lambda x:int(x))
    f2l={n:i for i,n in enumerate(classes)}; paths=[]; labels=[]
    for cls in classes:
        for fn in os.listdir(Path(root)/cls):
            p=Path(root)/cls/fn
            if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}:
                paths.append(str(p)); labels.append(f2l[cls])
    return paths, np.array(labels), classes

paths, yids, classes = load_raw_paths(DATASET_ROOT)
print(f"images={len(paths)} classes={len(classes)}")

embs=[]; miss=0
for p in paths:
    img = cv2.imread(p)                                   # BGR
    faces = app.get(img) if img is not None else []
    if faces:
        faces.sort(key=lambda f:(f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]), reverse=True)
        embs.append(faces[0].normed_embedding)
    else:                                                 # no detection -> feed whole resized image
        e = rec.get_feat(cv2.resize(img,(112,112)))[0] if img is not None else np.zeros(512,np.float32)
        embs.append(e); miss+=1
emb = np.asarray(embs, np.float32)
print(f"(no-detect fallback on {miss}/{len(paths)} images)")

def cv_knn(emb, yids, k=5):
    emb = emb/(np.linalg.norm(emb,axis=1,keepdims=True)+1e-9)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42); accs=[]
    for tr,vl in skf.split(emb,yids):
        G,gy=emb[tr],yids[tr]; Q,qy=emb[vl],yids[vl]; S=Q@G.T
        nn=np.argsort(-S,axis=1)[:,:k]; pred=[]
        for row,idx in zip(S,nn):
            sc={}
            for l,w in zip(gy[idx],row[idx]): sc[l]=sc.get(l,0.0)+w
            pred.append(max(sc,key=sc.get))
        accs.append(accuracy_score(qy,pred))
    return float(np.mean(accs)), float(np.std(accs)), [round(a*100,1) for a in accs]

m,s,f = cv_knn(emb, yids, k=5)
print(f"\n[InsightFace ArcFace buffalo_l, frozen + cosine kNN]  {m*100:.2f}% +/- {s*100:.2f}%  folds={f}")
print(f"[HayderVGR (ours, task-adapted, fine-tuned)        ]  97.45% +/- 0.98%   [from CV cell]")


In [ ]:
# ============================================================
# [Comparison 2] LFW public benchmark - verification (standard 10-fold pairs)
# PRIMARY: HayderVGR BNNeck embedder (TensorFlow, our trained model).
# OPTIONAL reference: InsightFace ArcFace (best-effort; skipped if it fails).
# Protocol: cosine similarity per pair; threshold fit on 9 folds, tested on the
# held-out fold; report acc mean+/-std + AUC.
# HONEST framing: HayderVGR is a CLOSED-SET student model; its LFW score is a
# transfer/generalization probe for external comparison, not its target objective.
# Run AFTER the algorithm cell (needs trained 'model', build_embedder_from_model).
# ============================================================
import numpy as np, tensorflow as tf
from sklearn.datasets import fetch_lfw_pairs
from sklearn.metrics import roc_auc_score

lfw = fetch_lfw_pairs(subset='10_folds', color=True, resize=1.0, funneled=True)
pairs = lfw.pairs.astype('float32'); labels = lfw.target.astype(int)
if pairs.max() > 1.5: pairs /= 255.0
N = len(labels); print(f"LFW pairs={N} | image shape={pairs.shape[2:]} | pos={int(labels.sum())}")

def lfw_eval(sims, labels, n_folds=10):
    fold=N//n_folds; accs=[]
    for f in range(n_folds):
        te=np.zeros(N,bool); te[f*fold:(f+1)*fold]=True; trn=~te
        ba,bt=0.0,0.0
        for t in np.unique(sims[trn]):
            a=((sims[trn]>=t)==labels[trn].astype(bool)).mean()
            if a>ba: ba,bt=a,t
        accs.append(((sims[te]>=bt)==labels[te].astype(bool)).mean())
    return float(np.mean(accs)), float(np.std(accs)), float(roc_auc_score(labels,sims))

# ---- (a) HayderVGR embedder (VGGFace preprocessing) ----
emb_model = build_embedder_from_model(model)
VGG_MEAN = np.array([93.5940,104.7624,129.1863], np.float32)   # BGR
def prep(b):
    x = tf.image.resize(b,(IMG_H,IMG_W)).numpy()*255.0
    return (x[...,::-1]-VGG_MEAN).astype(np.float32)
def h_sims(pairs):
    A=prep(np.stack([p[0] for p in pairs])); B=prep(np.stack([p[1] for p in pairs]))
    ea=emb_model.predict(A,batch_size=64,verbose=0); eb=emb_model.predict(B,batch_size=64,verbose=0)
    ea/= (np.linalg.norm(ea,axis=1,keepdims=True)+1e-9); eb/= (np.linalg.norm(eb,axis=1,keepdims=True)+1e-9)
    return np.sum(ea*eb,axis=1)
print("Embedding LFW with HayderVGR...")
sh=h_sims(pairs); mh,dh,ah=lfw_eval(sh,labels)
print(f"[HayderVGR embedding -> LFW]  acc {mh*100:.2f}% +/- {dh*100:.2f}%  | AUC {ah:.4f}")

# ---- (b) OPTIONAL InsightFace ArcFace reference ----
try:
    import subprocess, sys, cv2
    subprocess.run([sys.executable,"-m","pip","-q","install","insightface","onnxruntime-gpu"], check=False)
    from insightface.app import FaceAnalysis
    app=FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider','CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640,640)); rec=app.models['recognition']
    def i_feat(imgs01):                                   # RGB [0,1] -> BGR uint8 112 -> ArcFace
        out=[]
        for im in imgs01:
            bgr=cv2.resize((im[...,::-1]*255).astype('uint8'),(112,112))
            out.append(rec.get_feat(bgr)[0])
        e=np.asarray(out,np.float32); return e/(np.linalg.norm(e,axis=1,keepdims=True)+1e-9)
    ea=i_feat([p[0] for p in pairs]); eb=i_feat([p[1] for p in pairs])
    si=np.sum(ea*eb,axis=1); mi,di,ai=lfw_eval(si,labels)
    print(f"[InsightFace ArcFace -> LFW]  acc {mi*100:.2f}% +/- {di*100:.2f}%  | AUC {ai:.4f}")
except Exception as e:
    print(f"[InsightFace LFW reference skipped: {type(e).__name__}: {e}]")

print("\nNote: dedicated face models trained on millions of identities typically score ~99% on")
print("LFW; HayderVGR is optimized for the closed-set institutional task, so its LFW number")
print("reflects embedding transfer to unseen identities (external-comparison probe).")


In [ ]:
# ============================================================
# [Comparison 3] ResNet50 baseline RE-VALIDATION (rebut the ~36% anomaly)
# The earlier "ResNet50 + BNNeck + ArcFace -> 36%" almost certainly came from
# applying a LARGE fixed angular margin from epoch 1 on a cold backbone, which
# collapses ArcFace training. Here we re-run the SAME head with the corrected
# recipe used for HayderVGR: a MARGIN WARMUP (0.0 -> 0.3), correct ResNet50
# preprocessing, and progressive unfreezing -> a sane, fair baseline number.
# Protocol: 3-fold StratifiedKFold(seed=42). Reuses ArcFaceLoss / CosineDenseFixed /
# build_tfdata. Run AFTER the algorithm cell. (Trains 3 ResNet50 models -> a few min.)
# ============================================================
import os, numpy as np, tensorflow as tf, gc
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre
from tensorflow.keras.layers import (Input, GlobalAveragePooling2D, Dense,
                                     BatchNormalization, RandomRotation, RandomTranslation)
from tensorflow.keras.models import Model

# ---- load RAW images (RGB, [0,255]) in load_dataset order; apply ResNet50 preprocessing ----
def load_resnet_arrays(root, size):
    classes = sorted([d.name for d in Path(root).iterdir() if d.is_dir() and d.name.isdigit()],
                     key=lambda x:int(x))
    f2l = {n:i for i,n in enumerate(classes)}; X=[]; y=[]
    for cls in classes:
        for fn in os.listdir(Path(root)/cls):
            p = Path(root)/cls/fn
            if p.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp','.webp'}: continue
            img = tf.io.decode_image(tf.io.read_file(str(p)), channels=3, expand_animations=False)
            img = tf.image.resize(img, size); X.append(img.numpy()); y.append(f2l[cls])
    X = resnet_pre(np.asarray(X, dtype=np.float32))            # RGB[0,255] -> caffe BGR mean-sub
    C = len(classes); Y = np.eye(C, dtype=np.float32)[np.asarray(y)]
    return X, Y, np.asarray(y), C

Xr, Yr, yids_r, C = load_resnet_arrays(DATASET_ROOT, (IMG_H, IMG_W))
print(f"ResNet50 baseline data: images={len(Xr)} classes={C}")

def build_resnet_head(C):
    inp = Input(shape=(IMG_H, IMG_W, 3))
    x = RandomRotation(0.03)(inp); x = RandomTranslation(0.02,0.02)(x)
    base = ResNet50(include_top=False, weights='imagenet', input_tensor=x)
    x = GlobalAveragePooling2D(name='gap')(base.output)
    x = Dense(256, activation='relu', name='fc1')(x)
    x = BatchNormalization(scale=False, name='bnneck')(x)
    logits = CosineDenseFixed(C, s=ARC_S, name='cosine_dense')(x)
    return Model(inp, logits, name='resnet50_bnneck_arcface'), base

# margin warmup + progressive unfreeze (the fix vs the original collapse)
PHASES = [dict(m=0.00, ep=4,  lr=1e-3, unfreeze=False),   # warm up with NO margin
          dict(m=0.20, ep=6,  lr=5e-4, unfreeze=False),
          dict(m=0.30, ep=12, lr=1e-4, unfreeze=True)]

def set_trainable(base, unfreeze):
    for l in base.layers:
        l.trainable = bool(unfreeze and ('conv5' in l.name)
                           and not isinstance(l, tf.keras.layers.BatchNormalization))

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
accs=[]
for fold,(tr,vl) in enumerate(skf.split(Xr, yids_r),1):
    print("="*64); print(f"[ResNet50 re-validation] FOLD {fold}/3"); print("="*64)
    fix_seeds(42); tf.keras.backend.clear_session()
    x_tr,x_vl = Xr[tr],Xr[vl]; y_tr,y_vl = Yr[tr],Yr[vl]
    rmodel, base = build_resnet_head(C)
    spe = max(1, len(x_tr)//BATCH_SIZE)
    dtr = build_tfdata(x_tr, y_tr, BATCH_SIZE, training=True,  shuffle=True,  mixup_alpha=0.2)
    dvl = build_tfdata(x_vl, y_vl, BATCH_SIZE, training=False, shuffle=False, mixup_alpha=0.0)
    for ph in PHASES:
        set_trainable(base, ph['unfreeze'])
        rmodel.compile(optimizer=tf.keras.optimizers.Adam(ph['lr']),
                       loss=ArcFaceLoss(s=ARC_S, m=ph['m']), metrics=['accuracy'])
        rmodel.fit(dtr, validation_data=dvl, epochs=ph['ep'],
                   steps_per_epoch=spe, verbose=2)
    p = np.argmax(rmodel.predict(x_vl, batch_size=64, verbose=0), axis=1)
    a = accuracy_score(np.argmax(y_vl,axis=1), p); accs.append(a)
    print(f"  fold {fold} val-acc = {a*100:.2f}%")
    del rmodel, base; gc.collect()

print("\n" + "="*64)
print(f"[ResNet50 + BNNeck + ArcFace, corrected recipe]  {np.mean(accs)*100:.2f}% +/- {np.std(accs)*100:.2f}%")
print(f"  folds = {[round(a*100,1) for a in accs]}")
print("  -> Far above the earlier 36%, confirming that figure was a training")
print("     misconfiguration (large fixed margin on a cold backbone), not the architecture.")
print(f"[HayderVGR (VGGFace VGG16, ours)              ]  97.45% +/- 0.98%  [from CV cell]")


In [ ]:
# ============================================================
# [Figures] HayderVGR learning curves (Fig. 6) + quantitative ERROR ANALYSIS (R2#12)
# - Learning curves are loaded from the per-phase CSVs saved by train_phases (no re-train).
# - Error analysis is QUANTITATIVE only (confusion matrix with numeric IDs, most-confused
#   identity pairs, per-class accuracy and confidence distributions). Student face images
#   are deliberately NOT displayed, to respect the dataset's consent/numeric-label policy.
# Uses globals: model, x_val, y_val, dirs, f2l, EPOCHS_PHASES.  Run AFTER the algorithm cell.
# ============================================================
import os, csv, numpy as np, matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import confusion_matrix

curv = Path(dirs["curves"]); rep = Path(dirs["reports"]); rep.mkdir(parents=True, exist_ok=True)
print("curves dir:", curv)
try: print("available curve files:", sorted(os.listdir(curv))[:12])
except Exception: pass

# ---------- (1) Learning curves: concatenate phase1..4 ----------
def load_metric(names):
    if isinstance(names,str): names=[names]
    for nm in names:
        ys=[]
        for ph in ["phase1","phase2","phase3","phase4"]:
            f=curv/f"{ph}_{nm}.csv"
            if f.exists():
                with open(f) as fh:
                    next(fh,None)
                    ys += [float(l.strip().split(",")[1]) for l in fh if l.strip()]
        if ys: return ys
    return []
try:
    acc =load_metric(["accuracy","categorical_accuracy"])
    vacc=load_metric(["val_accuracy","val_categorical_accuracy"])
    loss=load_metric("loss"); vloss=load_metric("val_loss")
    if acc and vacc:
        fig,ax=plt.subplots(1,2,figsize=(11,4))
        ax[0].plot(range(1,len(acc)+1),acc,label="train"); ax[0].plot(range(1,len(vacc)+1),vacc,label="val")
        ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("accuracy"); ax[0].legend(); ax[0].grid(alpha=.3)
        if loss and vloss:
            ax[1].plot(range(1,len(loss)+1),loss,label="train"); ax[1].plot(range(1,len(vloss)+1),vloss,label="val")
            ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("loss"); ax[1].legend(); ax[1].grid(alpha=.3)
        for x in np.cumsum(EPOCHS_PHASES)[:-1]:
            for a in ax: a.axvline(x+0.5,color='gray',ls='--',lw=.8,alpha=.6)
        plt.suptitle("HayderVGR training curves (four-phase schedule)")
        plt.tight_layout(); p=rep/"haydervgr_learning_curves.png"; plt.savefig(p,dpi=300,bbox_inches='tight'); plt.close()
        print("saved learning curves ->", p)
    else:
        print("accuracy curve CSVs not found - skipping learning-curve figure")
except Exception as e:
    print("learning-curve plotting skipped:", type(e).__name__, e)

# ---------- predictions on validation set ----------
ytrue = np.argmax(y_val,axis=1) if np.ndim(y_val)>1 else np.asarray(y_val)
logits = model.predict(x_val, batch_size=32, verbose=0)
ypred  = np.argmax(logits, axis=1)
print(f"\nvalidation dense accuracy = {(ypred==ytrue).mean()*100:.2f}%  (n={len(ytrue)})")
inv = {v:k for k,v in f2l.items()} if 'f2l' in globals() else {}

# ---------- (2) confusion matrix (numeric IDs, no annotations) ----------
C = confusion_matrix(ytrue, ypred)
plt.figure(figsize=(7,6)); plt.imshow(C, cmap='viridis'); plt.colorbar(label='count')
plt.title(f"Confusion matrix ({C.shape[0]} identities)"); plt.xlabel("predicted ID"); plt.ylabel("true ID")
plt.tight_layout(); pcm=rep/"confusion_matrix.png"; plt.savefig(pcm,dpi=300,bbox_inches='tight'); plt.close()
print("saved confusion matrix ->", pcm)

# ---------- (3) top most-confused identity pairs ----------
Coff=C.copy(); np.fill_diagonal(Coff,0)
order=np.dstack(np.unravel_index(np.argsort(-Coff,axis=None), C.shape))[0]
print("\nTop misclassified identity pairs (true -> pred : count):")
shown=0
for t,p in order:
    if Coff[t,p]<=0: break
    print(f"  {inv.get(t,t)} -> {inv.get(p,p)} : {int(Coff[t,p])}"); shown+=1
    if shown>=10: break
if shown==0: print("  (no confusions - perfect on this split)")

# ---------- (4) per-class accuracy + confidence distributions ----------
row=C.sum(axis=1); per_cls=np.divide(np.diag(C), np.maximum(row,1))
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(per_cls*100, bins=20, color='#4878a8', edgecolor='white')
ax[0].set_title("Per-identity accuracy distribution"); ax[0].set_xlabel("accuracy (%)"); ax[0].set_ylabel("# identities"); ax[0].grid(alpha=.3)
ex=np.exp(logits-logits.max(axis=1,keepdims=True)); conf=(ex/ex.sum(axis=1,keepdims=True)).max(axis=1)
ok=ypred==ytrue
ax[1].hist(conf[ok], bins=20, alpha=.7, label="correct", color='#5a9e6f')
ax[1].hist(conf[~ok],bins=20, alpha=.7, label="incorrect", color='#c45b5b')
ax[1].set_title("Prediction confidence"); ax[1].set_xlabel("softmax confidence"); ax[1].set_ylabel("count"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); pca=rep/"error_analysis_distributions.png"; plt.savefig(pca,dpi=300,bbox_inches='tight'); plt.close()
print("saved error-analysis distributions ->", pca)
worst=np.argsort(per_cls)[:5]
print("lowest per-identity accuracies:", [(inv.get(int(c),int(c)), round(per_cls[c]*100,1)) for c in worst])
print("\nDownload the PNGs from:", rep)
